In [10]:
import os
from pathlib import Path
from time import sleep
from dotenv import load_dotenv
from openai import OpenAI
import re 


In [11]:
load_dotenv()
Api_key = os.getenv("XKRIO_API_KEY")
Base_url = os.getenv("XKIRO_BASE_URL")

if not Api_key or not Base_url:
    raise ValueError("api key not found...")

In [12]:
client = OpenAI(api_key=Api_key, base_url=Base_url)
model = "deepseek/deepseek-v4.1-flash:free" or "openai/gpt-5.3-codex-spark"
Role = "user"

In [13]:
def get_product_price(product):
    if product == "iPhone 17":
        return 1000
    elif product == "iPhone 18":
        return 500
    else:
        return 0
    

In [14]:
def calculator(expression):
    try:
        return eval(expression)
    except:
        return "calc error!"
    

In [15]:
tools = {
    "get_product_price": get_product_price,
    "calculator":calculator
}

In [16]:
system_prompt = """
You are a shopping assistant.

You have these tools:

get_product_price(product)
calculator(expression)
IMPORTANT:
Call tools exactly like these examples:

Action: get_product_price("iPhone 17")
Action: calculator("5000 - 1000")

Never write:
get_product_price(product="iPhone 17")

Never write:
calculator(expression="5000 - 1000")


When the task is complete, give the Final Answer.
"""

In [17]:
def run_agent(question):
    messages = [
        {
            "role":"system",
            "content":system_prompt
        },
        {
            "role": Role,
            "content": question
        }
    ]
    step = 1

    while True:
        print("\n------------------")
        print("STEP", step)
        print("------------------")

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0
        )

        answer = response.choices[0].message.content
        print(answer)

        if "Final Answer:" in answer:
            break 

        match = re.search(
            r"Action:\s*(\w+)\((.*?)\)",
            answer
        )

        if match:
            tool_name = match.group(1)
            tool_input = match.group(2)
            tool_input = tool_input.strip()
            tool_input = tool_input.strip('"')

            if tool_name in tools:

                tool = tools[tool_name]
                observation = tool(tool_input)

            else:
                observation = "Tool not found"

            print(
                "Observation:",
                observation
            )

            messages.append({
                "role":"assistant",
                "content":answer
            })

            messages.append({
                "role":Role,
                "content":"observation: "+ str(observation)
            })
            step+=1
            sleep(5)

In [18]:
prompt = """ 
I have 5000 rupees. What is the price of an iphone 17?
and how much money will I have left?
"""

run_agent(prompt)


------------------
STEP 1
------------------
Action: get_product_price("iPhone 17")
Observation: 1000

------------------
STEP 2
------------------
Action: calculator("5000 - 1000")
Observation: 4000

------------------
STEP 3
------------------
Final Answer: The price of an iPhone 17 is 1000 rupees. You will have 4000 rupees left.
